# Notebook 5 — Hyperparameter Optimization

## Objective

The baseline RT-DETR-L model achieved promising performance on the processed pothole dataset.

Although the baseline model demonstrated strong detection capability, several observations from the error analysis indicate room for improvement, particularly in detecting small potholes and increasing overall recall.

This notebook systematically investigates the effect of different hyperparameters on model performance.

Unlike arbitrary parameter tuning, each experiment modifies only a single training variable while keeping the remaining configuration identical to the baseline. This controlled approach enables objective evaluation of how individual hyperparameters influence detection accuracy and model generalization.

Each experiment will be compared against the baseline using identical evaluation metrics.

---

## Planned Experiments

| Experiment | Parameter Changed | Purpose |
|------------|------------------|----------|
| Baseline | None | Reference model |
| Experiment 1 | Image Size (640 → 800) | Improve small pothole detection |
| Experiment 2 | Epochs | Study convergence |
| Experiment 3 | Data Augmentation | Improve robustness |
| Experiment 4 | Learning Rate | Improve optimization |
| Experiment 5 | Best Combination | Final optimized model |

In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.8 MB/s eta 0:00:00a 0:00:01


In [18]:
import os
import json
import yaml
import shutil
import random
import time

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from ultralytics import RTDETR

## Experiment 1 — Higher Input Resolution

### Research Hypothesis

The exploratory data analysis revealed that a considerable proportion of potholes occupy a relatively small image area.

Increasing the input image resolution from **640×640** to **800×800** should provide the detector with finer spatial details, enabling improved localization of small potholes.

### Expected Outcome

- Higher Recall
- Higher mAP50-95
- Better localization

Potential drawbacks include:

- Increased GPU memory consumption
- Longer training time

In [19]:
# ============================================================
# Dataset Configuration
# ============================================================

from pathlib import Path
import yaml

# Path to Kaggle dataset
DATASET = Path(
    "/kaggle/input/datasets/krishnatheachiever/processed-pothole-yolo-dataset"
)

yaml_path = DATASET / "dataset.yaml"

# Load original YAML
with open(yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

# Update dataset root for current environment
cfg["path"] = str(DATASET)

# Save corrected YAML
FIXED_YAML = Path("/kaggle/working/dataset_fixed.yaml")

with open(FIXED_YAML, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print("=" * 60)
print("Dataset configuration generated successfully.")
print("=" * 60)
print(f"Dataset Root : {DATASET}")
print(f"YAML File    : {FIXED_YAML}")
print("=" * 60)

Dataset configuration generated successfully.
Dataset Root : /kaggle/input/datasets/krishnatheachiever/processed-pothole-yolo-dataset
YAML File    : /kaggle/working/dataset_fixed.yaml


In [20]:
# ============================================================
# Verify Dataset Configuration
# ============================================================

with open(FIXED_YAML) as f:
    print(f.read())

path: /kaggle/input/datasets/krishnatheachiever/processed-pothole-yolo-dataset
train: images/train
val: images/val
names:
  0: pothole



In [21]:
EXPERIMENT = {

    "experiment_name": "Exp01_ImageSize800",

    "model": "/kaggle/input/models/krishnatheachiever/rtdetr-baseline-experiment/pytorch/default/1/RTDETR_Baseline_Experiment/weights/best.pt",

    "data": str(FIXED_YAML),

    "epochs": 30,

    "imgsz": 800,

    "batch": 16,

    "optimizer": "AdamW",

    "lr0": 0.0005,

    "weight_decay": 0.0005,

    "workers": 4,

    "patience": 10,

    "device": 0,

    "seed": 42,

    "project": "RTDETR_Model_Refinement",

    "name": "Exp01_ImageSize800",

    "amp": True

}

In [22]:
pd.DataFrame(EXPERIMENT.items(), columns=["Parameter","Value"])

,Parameter,Value
0,experiment_name,Exp01_ImageSize800
1,model,/kaggle/input/models/krishnatheachiever/rtdetr...
2,data,/kaggle/working/dataset_fixed.yaml
3,epochs,30
4,imgsz,800
5,batch,16
6,optimizer,AdamW
7,lr0,0.0005
8,weight_decay,0.0005
9,workers,4


## Training Configuration

Only the image resolution differs from the baseline.

All remaining parameters remain unchanged to ensure that performance differences can be attributed solely to the increase in input resolution.

In [23]:
model = RTDETR(EXPERIMENT["model"])

In [24]:
from ultralytics import RTDETR

model = RTDETR(EXPERIMENT["model"])

print("Loaded trained baseline model successfully.")

Loaded trained baseline model successfully.


## Model Training

The RT-DETR-L model is trained using the configuration defined for **Experiment 1**.

This experiment differs from the baseline only in the input image resolution (**640 → 800 pixels**). All other hyperparameters remain unchanged to ensure that any observed performance differences can be attributed solely to the increased input resolution.

During training, the framework automatically performs validation after each epoch, saves the best-performing model checkpoint, records training metrics, and generates visualizations required for subsequent evaluation.

In [25]:
# ============================================================
# Start Training
# ============================================================

import time

print("=" * 60)
print(f"Starting {EXPERIMENT['experiment_name']}")
print("=" * 60)

start_time = time.time()

results = model.train(

    data=EXPERIMENT["data"],

    epochs=EXPERIMENT["epochs"],

    imgsz=EXPERIMENT["imgsz"],

    batch=EXPERIMENT["batch"],

    optimizer=EXPERIMENT["optimizer"],

    lr0=EXPERIMENT["lr0"],

    weight_decay=EXPERIMENT["weight_decay"],

    workers=EXPERIMENT["workers"],

    patience=EXPERIMENT["patience"],

    device=EXPERIMENT["device"],

    seed=EXPERIMENT["seed"],

    amp=EXPERIMENT["amp"],

    project=EXPERIMENT["project"],

    name=EXPERIMENT["name"],

    verbose=True,

    plots=True,

    save=True,

    val=True,

    save_period=-1

)

end_time = time.time()

training_time = end_time - start_time

hours = int(training_time // 3600)
minutes = int((training_time % 3600) // 60)
seconds = int(training_time % 60)

print("\n" + "=" * 60)
print("Training Completed")
print("=" * 60)
print(f"Experiment       : {EXPERIMENT['experiment_name']}")
print(f"Training Time    : {hours:02d}:{minutes:02d}:{seconds:02d}")
print(f"Average/Epoch    : {training_time / EXPERIMENT['epochs']:.2f} sec")
print("=" * 60)

Starting Exp01_ImageSize800
Ultralytics 8.4.107 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/models/krishnatheachiever/rtdetr-baseline-ex

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/30      8.46G      1.205     0.6739     0.5995         16        800: 100% ━━━━━━━━━━━━ 1250/1250 2.0it/s 10:210.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.2s/it 31.7s1.1ss
                   all        849       2080      0.206      0.375      0.199     0.0734

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/30      4.67G     0.8584     0.9335     0.2833          9        800: 0% ──────────── 0/1250  0.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/30      4.69G     0.7459     0.8415     0.3111         22        800: 100% ━━━━━━━━━━━━ 1250/1250 2.0it/s 10:140.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.0s/it 28.3s1.1ss
                   all        849       2080      0.388      0.444      0.353      0.142

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/30      5.38G     0.6925     0.8346     0.3811         11        800: 0% ──────────── 0/1250  0.5s

...

      28/30      5.38G     0.4196     0.4977     0.2028          7        800: 100% ━━━━━━━━━━━━ 1250/1250 2.1it/s 10:070.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.0s/it 28.3s1.1ss
                   all        849       2080      0.865      0.764      0.834      0.512

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      29/30      5.38G     0.5639     0.4402     0.2945          5        800: 0% ──────────── 0/1250  0.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      29/30      5.38G     0.4162     0.4878     0.2021         11        800: 100% ━━━━━━━━━━━━ 1250/1250 2.1it/s 10:060.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.0s/it 28.3s1.1ss
                   all        849       2080      0.844      0.786      0.822      0.503

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      30/30      5.38G     0.2243      0.428     0.2592          6        800: 0% ──────────── 0/1250  0.5s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      30/30      5.38G     0.4155     0.4908     0.2029          5        800: 100% ━━━━━━━━━━━━ 1250/1250 2.1it/s 10:080.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 1.0s/it 28.3s1.1ss
                   all        849       2080       0.85      0.786      0.835      0.508

30 epochs completed in 5.342 hours.
Optimizer stripped from /kaggle/working/runs/detect/RTDETR_Model_Refinement/Exp01_ImageSize800/weights/last.pt, 66.3MB
Optimizer stripped from /kaggle/working/runs/detect/RTDETR_Model_Refinement/Exp01_ImageSize800/weights/best.pt, 66.3MB

Validating /kaggle/working/runs/detect/RTDETR_Model_Refinement/Exp01_ImageSize800/weights/best.pt...
Ultralytics 8.4.107 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary: 315 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/2

In [31]:
# ============================================================
# Experiment Output Directory
# ============================================================

from pathlib import Path

SAVE_DIR = Path(results.save_dir)

print("=" * 60)
print("Experiment Output Directory")
print("=" * 60)
print(SAVE_DIR)
print("=" * 60)

Experiment Output Directory
/kaggle/working/runs/detect/RTDETR_Model_Refinement/Exp01_ImageSize800


In [32]:
#Experiment Comparison Table
import pandas as pd

# ============================================================
# Master Experiment Comparison Table
# ============================================================

comparison_df = pd.DataFrame({

    "Stage": [
        "Baseline",
        "Exp01",
        "Exp02",
        "Exp03",
        "Exp04",
        "Final"
    ],

    "Starting Model": [
        "COCO Pretrained",
        "Baseline best.pt",
        "Exp01 best.pt",
        "Exp02 best.pt",
        "Exp03 best.pt",
        "Best Model"
    ],

    "Modification": [
        "Initial Training",
        "Image Size = 800",
        "Data Augmentation",
        "Learning Rate Fine-Tuning",
        "Additional Optimization",
        "Final Optimized Model"
    ],

    "Epochs": [
        75,
        30,
        20,
        20,
        20,
        "-"
    ],

    "Precision": [
        0.891,
        None,
        None,
        None,
        None,
        None
    ],

    "Recall": [
        0.792,
        None,
        None,
        None,
        None,
        None
    ],

    "mAP@50": [
        0.854,
        None,
        None,
        None,
        None,
        None
    ],

    "mAP@50-95": [
        0.541,
        None,
        None,
        None,
        None,
        None
    ],

    "Training Time": [
        "7h 24m",
        None,
        None,
        None,
        None,
        None
    ]

})

comparison_df

,Stage,Starting Model,Modification,Epochs,Precision,Recall,mAP@50,mAP@50-95,Training Time
0,Baseline,COCO Pretrained,Initial Training,75,0.891,0.792,0.854,0.541,7h 24m
1,Exp01,Baseline best.pt,Image Size = 800,30,NaN,NaN,NaN,NaN,None
2,Exp02,Exp01 best.pt,Data Augmentation,20,NaN,NaN,NaN,NaN,None
3,Exp03,Exp02 best.pt,Learning Rate Fine-Tuning,20,NaN,NaN,NaN,NaN,None
4,Exp04,Exp03 best.pt,Additional Optimization,20,NaN,NaN,NaN,NaN,None
5,Final,Best Model,Final Optimized Model,-,NaN,NaN,NaN,NaN,None


In [33]:
comparison_df.loc[
    comparison_df["Stage"] == "Exp01",
    [
        "Precision",
        "Recall",
        "mAP@50",
        "mAP@50-95",
        "Training Time"
    ]
] = [
    0.862,
    0.768,
    0.831,
    0.511,
    "5h 22m"
]

comparison_df

,Stage,Starting Model,Modification,Epochs,Precision,Recall,mAP@50,mAP@50-95,Training Time
0,Baseline,COCO Pretrained,Initial Training,75,0.891,0.792,0.854,0.541,7h 24m
1,Exp01,Baseline best.pt,Image Size = 800,30,0.862,0.768,0.831,0.511,5h 22m
2,Exp02,Exp01 best.pt,Data Augmentation,20,NaN,NaN,NaN,NaN,None
3,Exp03,Exp02 best.pt,Learning Rate Fine-Tuning,20,NaN,NaN,NaN,NaN,None
4,Exp04,Exp03 best.pt,Additional Optimization,20,NaN,NaN,NaN,NaN,None
5,Final,Best Model,Final Optimized Model,-,NaN,NaN,NaN,NaN,None


In [34]:
# ============================================================
# Save Comparison Table
# ============================================================

comparison_path = SAVE_DIR / "comparison_table.csv"

comparison_df.to_csv(
    comparison_path,
    index=False
)

print("Comparison table saved.")
print(comparison_path)

Comparison table saved.
/kaggle/working/runs/detect/RTDETR_Model_Refinement/Exp01_ImageSize800/comparison_table.csv


In [35]:
# ============================================================
# Save Experiment Summary
# ============================================================

summary = f"""
============================================================
Experiment Summary
============================================================

Experiment Name : {EXPERIMENT['experiment_name']}

Starting Model  : Baseline best.pt

Modification    : Image Size = 800

Epochs          : {EXPERIMENT['epochs']}

Image Size      : {EXPERIMENT['imgsz']}

Precision       : 0.862

Recall          : 0.768

mAP50           : 0.831

mAP50-95        : 0.511

Training Time   : 5h 22m

============================================================
"""

summary_file = SAVE_DIR / "experiment_summary.txt"

with open(summary_file, "w") as f:
    f.write(summary)

print(summary)


Experiment Summary

Experiment Name : Exp01_ImageSize800

Starting Model  : Baseline best.pt

Modification    : Image Size = 800

Epochs          : 30

Image Size      : 800

Precision       : 0.862

Recall          : 0.768

mAP50           : 0.831

mAP50-95        : 0.511

Training Time   : 5h 22m




In [36]:
# ============================================================
# Save Experiment Configuration
# ============================================================

import json

experiment_info = {

    "experiment": EXPERIMENT,

    "results": {

        "precision":0.862,

        "recall":0.768,

        "mAP50":0.831,

        "mAP50_95":0.511,

        "training_time":"5h 22m"

    }

}

with open(SAVE_DIR/"experiment.json","w") as f:

    json.dump(experiment_info,f,indent=4)

print("experiment.json saved.")

experiment.json saved.


In [37]:
# ============================================================
# Archive Experiment
# ============================================================

import shutil

zip_path = shutil.make_archive(

    str(SAVE_DIR),

    "zip",

    SAVE_DIR

)

print("="*60)
print("Experiment archived successfully.")
print("="*60)
print(zip_path)

Experiment archived successfully.
/kaggle/working/runs/detect/RTDETR_Model_Refinement/Exp01_ImageSize800.zip


In [38]:
# ============================================================
# Verify Archive
# ============================================================

from pathlib import Path

zip_file = Path(f"{SAVE_DIR}.zip")

print("Exists :", zip_file.exists())

print(
    "Size   :",
    round(zip_file.stat().st_size/(1024*1024),2),
    "MB"
)

Exists : True
Size   : 121.96 MB
